# GPT from Scratch — Decoder-Only Transformer

A standalone educational implementation of a small GPT-style autoregressive language model using PyTorch.

**Pipeline:** corpus loading → train/validation/test split → character tokenizer → batches → causal self-attention → Transformer blocks → training → evaluation → text generation → checkpoint saving.

This notebook is designed for learning the architecture, not for pretraining a production LLM. It runs with a built-in demonstration corpus, but meaningful results require a larger text corpus in `corpus.txt`.


## 1. Imports and configuration

The defaults are intentionally small so the notebook can be smoke-tested on CPU. If a GPU is available, the notebook will use it automatically.


In [1]:

from pathlib import Path
import math, random, copy, json
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# Model/training hyperparameters: small educational defaults.
BLOCK_SIZE = 96
BATCH_SIZE = 32
EMBED_DIM = 128
NUM_HEADS = 4
NUM_LAYERS = 2
DROPOUT = 0.1
LEARNING_RATE = 3e-4
EPOCHS = 500
EVAL_EVERY_EPOCH = True
GRAD_CLIP = 1.0

assert EMBED_DIM % NUM_HEADS == 0, "EMBED_DIM must be divisible by NUM_HEADS."


Device: cuda



## 2. Load and split the corpus

Upload a UTF-8 `corpus.txt` into the working directory to use your own text. Otherwise, the built-in demo corpus is used to make sure the notebook runs end to end.


In [2]:

CORPUS_PATH = Path("/content/corpus.txt")
DEMO_TEXT = """
Autoregressive models predict the next symbol using the symbols that came before it.
A Transformer language model uses causal self-attention to combine information from earlier positions.
Causal masking prevents a token from attending to future tokens during training.
The embedding layer maps token identifiers into learned vector representations.
Positional information helps the model distinguish the order of tokens in a sequence.
A feed-forward network transforms each position after attention mixes contextual information.
Residual connections and layer normalization help Transformer blocks train effectively.
Language models are trained by minimizing next-token cross entropy over many examples.
At generation time, the model repeatedly predicts and samples the next character.
Evaluation uses held-out data to estimate generalization rather than memorization.
Small educational models reveal the mechanics but do not have the capabilities of large pretrained models.
"""
if CORPUS_PATH.exists():
    raw_text = CORPUS_PATH.read_text(encoding="utf-8")
    corpus_source = str(CORPUS_PATH)
else:
    raw_text = DEMO_TEXT.strip() + "\n"
    corpus_source = "built-in demo corpus"

raw_text = raw_text.replace("\r\n", "\n").replace("\r", "\n")
if len(raw_text) < 400:
    raise ValueError("Corpus is too short. Add more text to corpus.txt (at least 400 characters).")

n = len(raw_text)
train_end, val_end = int(n * 0.80), int(n * 0.90)
train_text = raw_text[:train_end]
val_text = raw_text[train_end:val_end]
test_text = raw_text[val_end:]

# Fit the character vocabulary on the training split only.
special_tokens = ["<unk>"]
itos = special_tokens + sorted(set(train_text))
stoi = {ch: i for i, ch in enumerate(itos)}
UNK_ID = stoi["<unk>"]

def encode(s):
    return [stoi.get(ch, UNK_ID) for ch in s]

def decode(ids):
    return "".join(itos[i] if 0 <= i < len(itos) and i != UNK_ID else "�" for i in ids)

train_ids = torch.tensor(encode(train_text), dtype=torch.long)
val_ids = torch.tensor(encode(val_text), dtype=torch.long)
test_ids = torch.tensor(encode(test_text), dtype=torch.long)

print("Corpus:", corpus_source)
print(f"Characters: {n:,} | train={len(train_ids):,} | validation={len(val_ids):,} | test={len(test_ids):,}")
print("Vocabulary size:", len(itos))
if corpus_source == "built-in demo corpus":
    print("NOTE: Built-in data is only for pipeline verification. Use a larger corpus for meaningful experiments.")


Corpus: corpus.txt
Characters: 10,831 | train=8,664 | validation=1,083 | test=1,084
Vocabulary size: 114



## 3. Create batches for next-character prediction

Each input block is paired with a target shifted by one character. The model sees only the input prefix because self-attention is causally masked.


In [3]:

class NextCharDataset(Dataset):
    def __init__(self, ids, block_size):
        self.ids = ids
        self.block_size = block_size

    def __len__(self):
        return max(0, len(self.ids) - self.block_size)

    def __getitem__(self, idx):
        x = self.ids[idx:idx + self.block_size]
        y = self.ids[idx + 1:idx + self.block_size + 1]
        return x, y

train_ds = NextCharDataset(train_ids, BLOCK_SIZE)
val_ds = NextCharDataset(val_ids, BLOCK_SIZE)
test_ds = NextCharDataset(test_ids, BLOCK_SIZE)

if min(len(train_ds), len(val_ds), len(test_ds)) <= 0:
    raise ValueError("At least one split is shorter than BLOCK_SIZE; use more text or lower BLOCK_SIZE.")

def loader_for(ds, shuffle=False):
    return DataLoader(ds, batch_size=min(BATCH_SIZE, len(ds)), shuffle=shuffle, drop_last=False)

train_loader = loader_for(train_ds, shuffle=True)
val_loader = loader_for(val_ds)
test_loader = loader_for(test_ds)

xb, yb = next(iter(train_loader))
print("Input batch:", tuple(xb.shape), "| Target batch:", tuple(yb.shape))


Input batch: (32, 96) | Target batch: (32, 96)



## 4. Build the GPT architecture

Each Transformer block contains:
1. **Causal multi-head self-attention** — tokens attend only to themselves and earlier tokens.
2. **Residual connection + layer normalization**.
3. **Position-wise feed-forward network**.
4. Another residual connection and normalization.

The model combines token embeddings with learned positional embeddings, then predicts next-character logits.


In [4]:

class CausalSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, block_size, dropout):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.qkv = nn.Linear(embed_dim, 3 * embed_dim)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        # True means the location is allowed to attend. Lower-triangular = causal.
        self.register_buffer("causal_mask", torch.tril(torch.ones(block_size, block_size, dtype=torch.bool)))

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # Prefer PyTorch's optimized scaled-dot-product attention when available.
        if hasattr(F, "scaled_dot_product_attention"):
            out = F.scaled_dot_product_attention(
                q, k, v,
                attn_mask=self.causal_mask[:T, :T],
                dropout_p=self.attn_dropout.p if self.training else 0.0,
            )
        else:
            scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
            scores = scores.masked_fill(~self.causal_mask[:T, :T], float("-inf"))
            weights = self.attn_dropout(torch.softmax(scores, dim=-1))
            out = weights @ v

        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.proj(out))


class FeedForward(nn.Module):
    def __init__(self, embed_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.GELU(),
            nn.Linear(4 * embed_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = CausalSelfAttention(embed_dim, num_heads, block_size, dropout)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.ffwd = FeedForward(embed_dim, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(self, vocab_size, block_size, embed_dim, num_heads, num_layers, dropout):
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.position_embedding = nn.Embedding(block_size, embed_dim)
        self.blocks = nn.Sequential(*[
            TransformerBlock(embed_dim, num_heads, block_size, dropout)
            for _ in range(num_layers)
        ])
        self.final_ln = nn.LayerNorm(embed_dim)
        self.lm_head = nn.Linear(embed_dim, vocab_size)
        # Weight tying is a common GPT-family parameter-saving technique.
        self.lm_head.weight = self.token_embedding.weight

    def forward(self, idx, targets=None):
        B, T = idx.shape
        if T > self.block_size:
            raise ValueError(f"Sequence length {T} exceeds context length {self.block_size}.")
        positions = torch.arange(T, device=idx.device)
        x = self.token_embedding(idx) + self.position_embedding(positions)[None, :, :]
        x = self.blocks(x)
        logits = self.lm_head(self.final_ln(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, prompt, max_new_chars=300, temperature=0.8, top_k=None):
        self.eval()
        ids = encode(prompt)
        if not ids:
            ids = [stoi.get("\n", UNK_ID)]
        idx = torch.tensor([ids], dtype=torch.long, device=DEVICE)
        for _ in range(max_new_chars):
            context = idx[:, -self.block_size:]
            logits, _ = self(context)
            next_logits = logits[:, -1, :] / max(float(temperature), 1e-5)
            if top_k is not None:
                k = min(int(top_k), next_logits.size(-1))
                threshold = torch.topk(next_logits, k).values[:, [-1]]
                next_logits = next_logits.masked_fill(next_logits < threshold, float("-inf"))
            probs = F.softmax(next_logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return decode(idx[0].tolist())

model = MiniGPT(
    vocab_size=len(itos),
    block_size=BLOCK_SIZE,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
).to(DEVICE)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {num_params:,}")
logits, loss = model(xb.to(DEVICE), yb.to(DEVICE))
print("Logits shape:", tuple(logits.shape), "| initial loss:", round(loss.item(), 4))


Trainable parameters: 423,794
Logits shape: (32, 96, 114) | initial loss: 84.2858



## 5. Train the model

We select the best checkpoint by validation loss. Training loss typically falls first; if validation loss begins rising while training loss keeps falling, the model may be overfitting.


In [5]:

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, total_chars = 0.0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        _, loss = model(x, y)
        count = y.numel()
        total_loss += loss.item() * count
        total_chars += count
    avg_loss = total_loss / max(total_chars, 1)
    return avg_loss, math.exp(min(avg_loss, 20.0))

best_val_loss = float("inf")
best_state = None
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, total_chars = 0.0, 0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        _, loss = model(x, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        total_loss += loss.item() * y.numel()
        total_chars += y.numel()

    train_loss = total_loss / max(total_chars, 1)
    val_loss, val_ppl = evaluate(model, val_loader)
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_char_perplexity": val_ppl,
    })
    if epoch==1 or epoch%100==0 or epoch==EPOCHS:
      print(f"Epoch {epoch:02d}/{EPOCHS} | train loss={train_loss:.4f} | val loss={val_loss:.4f} | val char-PPL={val_ppl:.2f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())

if best_state is not None:
    model.load_state_dict(best_state)
    print(f"Restored best checkpoint (validation loss={best_val_loss:.4f}).")


Epoch 01/500 | train loss=10.5625 | val loss=2.9038 | val char-PPL=18.24
Epoch 100/500 | train loss=0.1142 | val loss=4.7863 | val char-PPL=119.85
Epoch 200/500 | train loss=0.0897 | val loss=5.2494 | val char-PPL=190.45
Epoch 300/500 | train loss=0.0822 | val loss=5.3204 | val char-PPL=204.46
Epoch 400/500 | train loss=0.0782 | val loss=5.3971 | val char-PPL=220.77
Epoch 500/500 | train loss=0.0759 | val loss=5.3695 | val char-PPL=214.75
Restored best checkpoint (validation loss=2.1370).



## 6. Test and generate text

Character-level perplexity measures how well the model predicts held-out characters. It is useful for comparing experiments on the same dataset, but it is not directly comparable to token-level perplexity.


In [6]:

test_loss, test_ppl = evaluate(model, test_loader)
print(f"Test cross-entropy: {test_loss:.4f}")
print(f"Test character-level perplexity: {test_ppl:.2f}")

prompt = "Naruto Uzumaki (うずまき ナルト) – a shinobi of Konohagakure (Hidden "
print("\nPrompt:", repr(prompt))
print("\nGenerated sample:\n")
print(model.generate(prompt, max_new_chars=400, temperature=0.8, top_k=10))


Test cross-entropy: 2.1079
Test character-level perplexity: 8.23

Prompt: 'Naruto Uzumaki (うずまき ナルト) – a shinobi of Konohagakure (Hidden '

Generated sample:

Naruto Uzumaki (うずまき ナルト) – a shinobi of Konohagakure (Hidden (langan) – adaring ninjaka a and reator chate in (Itachi covered parer I); Jiraiya; **Jura*; and bi/Fourto *Kishino** (from themeats and Marringa thentsuki) seaf; Naruto ento f illate in nallys sording of ristnatachais dreat forus com proshimones, switingedeptionces. Sen asuki and and Saringe mother them tof the Furinja. He *Sinals ther *Jugairaina** (多重影分身の術) – nenting probogage), Naruto’s secrac



## 7. Save the checkpoint

The saved `.pt` checkpoint includes weights and architecture configuration. The JSON file records the vocabulary, hyperparameters, and training history.


In [7]:

checkpoint = {
    "model_state_dict": model.state_dict(),
    "vocab": itos,
    "block_size": BLOCK_SIZE,
    "embed_dim": EMBED_DIM,
    "num_heads": NUM_HEADS,
    "num_layers": NUM_LAYERS,
    "dropout": DROPOUT,
    "best_val_loss": best_val_loss,
}
torch.save(checkpoint, "04_mini_gpt_from_scratch.pt")
with open("04_mini_gpt_metadata.json", "w", encoding="utf-8") as f:
    json.dump({
        "vocab": itos,
        "block_size": BLOCK_SIZE,
        "embed_dim": EMBED_DIM,
        "num_heads": NUM_HEADS,
        "num_layers": NUM_LAYERS,
        "dropout": DROPOUT,
        "best_val_loss": best_val_loss,
        "history": history,
    }, f, ensure_ascii=False, indent=2)

print("Saved: 04_mini_gpt_from_scratch.pt")
print("Saved: 04_mini_gpt_metadata.json")


Saved: 04_mini_gpt_from_scratch.pt
Saved: 04_mini_gpt_metadata.json



## 8. Experiments

Change one variable at a time and rerun the notebook:
- `NUM_HEADS`: compare 2 vs. 4 while keeping `EMBED_DIM` divisible by the head count.
- `NUM_LAYERS`: compare 1 vs. 2 blocks.
- `BLOCK_SIZE`: test shorter or longer contexts.
- `DROPOUT`: try 0.0 vs. 0.1 on a sufficiently large corpus.
- `temperature` and `top_k`: compare output diversity and coherence.
- Inspect train/validation curves to identify underfitting or overfitting.

### What this implementation leaves out
This is a compact GPT-like model with learned positional embeddings. It does not implement RoPE, KV caching, distributed training, instruction tuning, or modern large-scale serving optimizations. Those are separate topics in the roadmap.
